# Importare le risonanze in python
Per importare i file NIfTI (lo standard nel neuroimaging) useremo la libreria *nibabel*

In [ ]:
# importa la libreria con l'alias nib
import nibabel as nib

# Load the NIfTI file
# Se il notebook è nella cartella giusta questo dovrebbe funzionare, altrimenti aggiusta il path
file_path = 'IXI_379/IXI379-Guys-0943-T1.nii.gz'
file_path = 'sub16ses2/anat/T1w1_denoise.nii.gz'
img = nib.load(file_path)
img

In [ ]:
print(img)

In [ ]:
# voxel dimensions
img.header.get_zooms()

In [ ]:
# units of measurement
img.header.get_xyzt_units()

In [ ]:
# shape of the image
img.shape

In [ ]:
# Extract the data as a numpy array
# get_fdata() converts the data to floating point numbers
data = img.get_fdata()
data

# Visualizzare la risonanza in python
## Plottare uno slice dell'immagine

In [ ]:
import matplotlib.pyplot as plt

# Select a slice to visualize
# We'll take a slice right through the middle of the third axis
middle_slice_idx = data.shape[2] // 2
slice_data = data[:, :, middle_slice_idx]

# Plot the slice
plt.figure(figsize=(6, 6))
# Using a grayscale colormap and rotating/transposing if necessary depending on the image orientation
plt.imshow(slice_data, cmap='gray') 
plt.title('T1 Slice')
plt.axis('off') # Hides the axis ticks
plt.show()

## Plottare tre slice ortogonali usando *nilearn*

In [ ]:
from nilearn import plotting

# Plot a static anatomical image
# This automatically displays 3 orthogonal cuts 
plotting.plot_anat(file_path, title='T1 Weighted Image')

plotting.show()

## Plot interattivo

In [ ]:
# Create the interactive view object
plotting.view_img(file_path, title = 'T1 image', cmap = 'Greys', threshold = 10, bg_img = None)

In [ ]:
help(plotting.view_img)

# Usare l'output di Freesurfer

### Calcolare il volume di una regione cerebrale
Cominciamo col vedere come fare a calcolare il volume di specifiche regioni celebrali.

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
import mne

In [ ]:
img = nib.load('sub16ses2/freesurfer/mri/aparc+aseg.mgz')

# mgz files are usually in mm, so this is in mm3
voxel_volume = np.prod(img.header.get_zooms())

# array of the atlas
atlas = img.get_fdata()

# we convert the numbers in the array to integers
atlas = atlas.astype(int)

In [ ]:
# let's see which labels are present in the atlas
np.unique(atlas)

In [ ]:
# lets compute the volume of the region corresponding to label 2

# this is a boolean mask, true in the voxels labelled with 2 and False otherwise
mask = (atlas == 2)

# let's plot it
plt.imshow(mask[:,100,:], cmap='gray') 

In [ ]:
# how many voxels are there labelled with 2?

# boolean mask follow specific rules when they are converted to numerical values:
# True becomes 1 and False becomes 0

# we can follow two equivalent approaches:
#   we can sum all the values
#   or we can count the values different from 0
print(np.sum(mask), np.count_nonzero(mask))

In [ ]:
# we have the number of voxels, to compute the volume we multiply it by the volume of a single voxel
number_of_voxels = np.sum(mask)

print(f'Label 2 has a volume of {voxel_volume*number_of_voxels} millimeters cubed.')

Bene, ora calcoliamolo per ogni regione del nostro atlante e assegnamo un nome a ogni valore.

In [ ]:
# load the dictionary that connects labels to names
fsLUT = mne.read_freesurfer_lut('./FreeSurferColorLUT.txt')[0]
fsLUT

In [ ]:
# now let's write a function that, given a label, returns the volume of the region
# note how the variable voxel_volume is not defined inside the function, so python automatically retrieves it from outside
def compute_volume(atlas, label):
    number_of_voxels = np.count_nonzero(atlas == label)
    return number_of_voxels*voxel_volume

print(compute_volume(atlas, 2))

In [ ]:
# now let's iterate over each element of our dictionary and compute the volume
# not the best approach, but it works

possible_values = np.unique(atlas)
for name, label in fsLUT.items():
    if label in possible_values:
        print(f'Region {name} has a volume of {compute_volume(atlas, label)} millimeters cubed.')

## Lavorare con le superfici di freesurfer
Il punto forte di freesurfer è la ricostruzione delle superfici della corteccia, vedremo come visualizzarle, e come calcolare l'area delle regioni in maniera accurata.

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import mne

### Plottare una superficie in python
Cominciamo con l'aprire queste superfici e plottarle interattivamente.

In [ ]:
pial_coords, pial_faces = nib.freesurfer.read_geometry('sub16ses2/freesurfer/surf/lh.pial')

# surfaces are often represented as "triangulated surfaces", i.e. surfaces made up of triangles
# you can think about them as a list of vertices (points in 3D space)
# and as a list of faces (indices of the vertices that make up each triangle)
pial_coords[:5], pial_faces[:4]

In [ ]:
# using pyvista we can plot them
# don't worry about the details here
def surface_to_pyvista(coords, faces):
    """This function converts a list of vertices and faces to a PyVista object for plotting"""
    faces_padded = np.c_[np.full(len(faces), 3), faces].ravel()
    mesh = pv.PolyData(coords, faces_padded)
    return mesh

pial_mesh = surface_to_pyvista(pial_coords, pial_faces)
pial_mesh.plot(color='green', smooth_shading=True)

In [ ]:
# now let's plot both the white and gray matter surfaces of both emispheres!

left_pial_coords, left_pial_faces = nib.freesurfer.read_geometry('sub16ses2/freesurfer/surf/lh.pial')
left_white_coords, left_white_faces = nib.freesurfer.read_geometry('sub16ses2/freesurfer/surf/lh.white')
right_pial_coords, right_pial_faces = nib.freesurfer.read_geometry('sub16ses2/freesurfer/surf/rh.pial')
right_white_coords, right_white_faces = nib.freesurfer.read_geometry('sub16ses2/freesurfer/surf/rh.white')

left_pial = surface_to_pyvista(left_pial_coords, left_pial_faces)
left_white = surface_to_pyvista(left_white_coords, left_white_faces)
right_pial = surface_to_pyvista(right_pial_coords, right_pial_faces)
right_white = surface_to_pyvista(right_white_coords, right_white_faces)


# Initialize a PyVista Plotter (Think of this as setting up a virtual studio)
plotter = pv.Plotter()

# Add the White matter (Opaque and white)
plotter.add_mesh(left_white, color='white', smooth_shading=True)
plotter.add_mesh(right_white, color='white', smooth_shading=True)

# Add the Pial matter (Pink-ish and transparent!)
plotter.add_mesh(left_pial, color='lightpink', opacity=0.8, smooth_shading=True)
plotter.add_mesh(right_pial, color='lightpink', opacity=0.8, smooth_shading=True)

# Set a nice background color and show the plot
plotter.set_background('slate_grey')
plotter.show()

### Visualizzare lo spessore della corteccia
Capire quanto è spessa la corteccia in ogni punto è difficile con la visualizzazione di sopra, proviamo ad aiutarci con un po' di colore.

In [ ]:
# Freesurfer estimates thickness at every vertex of the surface, we just need to load it from disk
left_thickness = nib.freesurfer.read_morph_data('sub16ses2/freesurfer/surf/lh.thickness')
right_thickness = nib.freesurfer.read_morph_data('sub16ses2/freesurfer/surf/rh.thickness')

# we assign the values directly to the PyVista mesh
left_white['Thickness'] = left_thickness
right_white['Thickness'] = right_thickness

plotter = pv.Plotter()

# Add the White matter (Opaque and white)
plotter.add_mesh(left_white, scalars='Thickness', cmap='coolwarm', smooth_shading=True)
plotter.add_mesh(right_white, scalars='Thickness', cmap='coolwarm', smooth_shading=True)

plotter.show()

### Visualizzare le regioni della corteccia
Un'altra cosa che possiamo fare è usare i colori per evidenziare le diverse regioni della corteccia.

In [ ]:
import matplotlib.colors as mcolors

# Load the Atlas Annotation File from freesurfer
atlas_file = 'sub16ses2/freesurfer/label/lh.aparc.annot'

# read_annot returns 3 things:
# - labels: An array of region IDs for every vertex
# - ctab: The official Color Table (RGBA + ID) for the atlas
# - names: The text names of the regions
labels, ctab, names = nib.freesurfer.read_annot(atlas_file)

# FreeSurfer stores names as byte-strings (e.g., b'precentral')
# We MUST decode them to normal Python strings first!
names = list(map(lambda x: x.decode('utf-8'), names))

# Assign the labels to our mesh
left_pial['Atlas_Regions'] = labels

# Now we Build a custom colormap using the official FreeSurfer colors!
# We extract the RGB columns from the ctab and normalize them from 0-255 to 0.0-1.0
official_colors = ctab[:, :3] / 255.0
fs_cmap = mcolors.ListedColormap(official_colors)

# let's build a legend
legend_entries = []
# Loop through the names and their corresponding colors
for i, name_str in enumerate(names):
    # Grab the RGB color for this specific region
    color = official_colors[i] 
    
    # PyVista expects a list of [["Name", color], ["Name", color]]
    legend_entries.append([name_str, color])

# Plot!
plotter = pv.Plotter()

# We tell PyVista to color based on our labels, using our custom FreeSurfer colormap
plotter.add_mesh(
    left_pial, 
    scalars='Atlas_Regions',
    cmap=fs_cmap, 
    smooth_shading=True,
    show_scalar_bar=False # Hides the legend since it's just numbers
)

# Add our custom legend to the plotter
# We tweak the size and font so all 35+ regions actually fit on the screen
plotter.add_legend(
    legend_entries, 
    bcolor=None,       # Transparent background for the legend box
    face='circle',     # Use little circles for the color swatches
    size=(0.15, 0.9),  # Width and Height of the legend box (scale of 0 to 1)
    loc='upper left'   # Pin it to the corner
)

plotter.show()

### Calcolare l'area di ogni regione
L'ultima cosa che vedremo è come calcolare l'area delle regioni. Questo valore è molto più indicativo del volume nella corteccia.

In [ ]:
# to compute the area, we either compute the area of each triangle manually
# or we use the estimate that freesurfer provides for each vertex
left_area = nib.freesurfer.read_morph_data('sub16ses2/freesurfer/surf/lh.area')

# let's start by computing the total area of the left emisphere
area_left_cortex = np.sum(left_area)
print(f'The total area of the left emisphere is {area_left_cortex} millimeters squared')

In [ ]:
# let's compute it region by region now
for region in np.unique(labels):
    region_area = np.sum(left_area[labels == region])
    print(f'The area of the left {names[region]} is {region_area:g} millimeters squared.')